<a href="https://colab.research.google.com/github/athfizh/PemMes26_05_Hafizh/blob/main/JS09/JS09-04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **Approximate Nearest Neighbors (ANN) - Lab 4**

---

> Lab 4 mengadu tiga library ANN sekaligus: **ANNOY**, **FAISS**, dan **HNSW**. Datanya 1.000.000 vektor 5D, query-nya satu vektor acak, dan kita ambil 10 tetangga terdekat. Yang dibandingkan adalah **build time** (waktu membangun index) dan **query time** (waktu mencari tetangga).

---

# **Langkah 0 - Install Library**

---

In [1]:
# Install library ANN yang dipakai.
# memakai faiss-gpu, kalau wheel-nya tidak cocok dengan
# versi Python Colab, otomatis fallback ke faiss-cpu.
!pip install -q annoy hnswlib
!pip install -q faiss-gpu

import subprocess, sys

try:
    import faiss
    print("FAISS terpasang:", faiss.__version__)
    print("GPU support:", hasattr(faiss, "StandardGpuResources"))
except Exception as e:
    print("faiss-gpu tidak bisa dipakai di runtime ini:", e)
    print("Fallback: memasang faiss-cpu supaya notebook tetap jalan...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=False)
    import faiss
    print("FAISS CPU terpasang:", faiss.__version__)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
FAISS terpasang: 1.15.1
GPU support: True


> Cell ini memasang <code>annoy</code>, <code>hnswlib</code>, dan <code>faiss-gpu</code> sekaligus. Bagian <code>try/except</code> hanya jaring pengaman: kalau <code>faiss-gpu</code> tidak tersedia untuk versi Python Colab, otomatis dipasang <code>faiss-cpu</code> supaya cell import di bawah tidak error.

---

# **Langkah 1 - Import Library**

---

In [2]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

> <code>numpy</code> dan <code>time</code> untuk data dan pengukuran, <code>AnnoyIndex</code> untuk ANNOY, <code>faiss</code> untuk FAISS, dan <code>hnswlib</code> untuk HNSW. Ketiganya sudah dipasang di Langkah 0, jadi cell ini tinggal import.

---

# **Langkah 2 - Build Dataset 1 Juta x 5D**

---

In [3]:
# ===============================
# Build 1mio rows with 5D
# ===============================
n_data = 1_000_000   # try 100_000 if you have limited memory
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

# Query point
query = np.random.random((1, dim)).astype(np.float32)
k = 10

print("Shape dataset:", X.shape)
print("Shape query  :", query.shape)

Shape dataset: (1000000, 5)
Shape query  : (1, 5)


> <code>np.random.random</code> membuat 1.000.000 baris dengan 5 fitur, semuanya <code>float32</code> supaya hemat memori dan cocok dengan API FAISS/HNSW. Satu juta kali lima kolom cuma sekitar 20 MB, jadi aman untuk Colab. Kalau RAM atau kuota Colab kamu terbatas, ganti <code>n_data</code> jadi <code>100_000</code> seperti saran di komentar modul.

---

# **Langkah 3 - ANNOY**

---

In [4]:
# ===============================
# 2. Annoy
# ===============================
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)  # 10 trees
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

=== Annoy ===
Build time: 27.06441307067871 detik
Query time: 0.00024008750915527344 detik
Neighbors: [328610, 929284, 921507, 884872, 562580] ...


> ANNOY membangun **10 pohon hyperplane acak**. Build time-nya lumayan lama karena harus memasukkan satu juta item satu per satu lewat loop Python, lalu membangun pohonnya. Tapi begitu index jadi, query-nya sangat cepat karena ANNOY hanya menyusuri leaf yang relevan.

---

# **Langkah 4 - FAISS (Flat Index)**

---

In [5]:
# ===============================
# 3. FAISS (Flat Index)
# ===============================
print("\n=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")


=== FAISS (IndexFlatL2) ===
Build time: 0.017632007598876953 detik
Query time: 0.011016368865966797 detik
Neighbors: [328610 929284 921507 707538 884872] ...


> <code>IndexFlatL2</code> adalah index **exact**: dia menyimpan semua vektor apa adanya dan menghitung jarak ke seluruh data saat query. Build time-nya hampir nol karena cuma menyalin data, tapi query time-nya lebih lambat daripada ANNOY/HNSW karena harus memindai satu juta vektor. Di lab ini FAISS Flat berperan sebagai **baseline exact**, bukan sebagai metode approximate.

---

# **Langkah 5 - HNSW (hnswlib)**

---

In [6]:
# ===============================
# 4. HNSW (hnswlib)
# ===============================
print("\n=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")


=== HNSW (hnswlib) ===
Build time: 182.9792730808258 detik
Query time: 0.00033593177795410156 detik
Neighbors: [328610 929284 921507 707538 884872] ...


> HNSW membangun graf bertingkat dengan <code>M=16</code> koneksi per node dan <code>ef_construction=200</code>. Build time-nya paling lama karena harus menyusun graf untuk satu juta node. Tapi setelah jadi, query-nya paling cepat: HNSW cukup berjalan dari level atas yang jarang, lalu turun ke level bawah sampai menemukan tetangga terdekat.

> Kesimpulan penting: **tidak ada pemenang mutlak**. ANNOY menang di kemudahan dan build yang moderat, FAISS Flat menang di kesederhanaan dan akurasi exact, HNSW menang di latensi query. Pilihan tergantung kebutuhan: apakah lebih penting build cepat, query cepat, atau akurasi?